In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Torch Gather

In [2]:
B, T, E = 2, 3, 4
weights = torch.tensor([
    [[ 0.0,  0.1,  0.2,  0.3], [ 1.0,  1.1,  1.2,  1.3], [ 2.0,  2.1,  2.2,  2.3]],
    [[10.0, 10.1, 10.2, 10.3], [11.0, 11.1, 11.2, 11.3], [12.0, 12.1, 12.2, 12.3]],
])
weights.shape

torch.Size([2, 3, 4])

In [3]:
K = 2
indices = torch.tensor([
    [[0, 1], [1, 2], [2, 3]],
    [[0, 1], [0, 2], [0, 3]],
])
indices.shape

torch.Size([2, 3, 2])

In [4]:
values = torch.gather(weights, dim=-1, index=indices)
values.shape

torch.Size([2, 3, 2])

In [5]:
values_check = torch.tensor([
    [[ 0.0,  0.1], [ 1.1,  1.2], [ 2.2,  2.3]],
    [[10.0, 10.1], [11.0, 11.2], [12.0, 12.3]],
])
values.equal(values_check)

True

## Step-by-Step Version

In [25]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)

B, T, C = 5, 16, 32
E = 4  # num experts
K = 2  # top_k

router = nn.Linear(C, E, bias=False)
experts_up = nn.ModuleList([nn.Linear(C, C*4, bias=False) for _ in range(E)])
experts_down = nn.ModuleList([nn.Linear(C*4, C, bias=False) for _ in range(E)])

In [26]:
x = torch.randn(B, T, C)
expert_bias = torch.tensor([0.0, 0.1, 0.2, 0.3])  # (E,)

In [27]:
x_flat = x.reshape(-1, C)          # B*T, C
logits = router(x_flat)       # B*T, E

In [28]:
weights = torch.sigmoid(logits)    # B*T, E

In [29]:
# Bias the expert selection, but *not* weighting (Nanochat, DeepSeekV3)
weights_biased = weights + expert_bias   # B*T, E

In [30]:
_, indices = torch.topk(weights_biased, K, dim=-1)     # B*T, K

In [31]:
values = torch.gather(weights, dim=-1, index=indices)   # B*T, K

In [32]:
x_flat_stacked = torch.stack([x_flat]*K, dim=1)      # B*T, K, C

In [33]:
x_flat_stacked_flat = x_flat_stacked.reshape(-1, C)  # B*T*K, C

In [34]:
indices_flat = indices.reshape(-1)                   # B*T*K

In [35]:
indices_flat_sorted_indices = torch.argsort(indices_flat, stable=True)  # B*T*K

In [36]:
x_flat_stacked_flat_sorted = x_flat_stacked_flat[indices_flat_sorted_indices]  # B*T*K, C

In [37]:
start_idx = 0
outs = []
for i in range(E):
    num_expert = (indices==i).sum().item()
    end_idx = start_idx + num_expert
    h = experts_up[i](x_flat_stacked_flat_sorted[start_idx:end_idx])
    z = F.relu(h).square()
    o = experts_down[i](z)
    outs.append(o)
    start_idx += num_expert
out_flat_stacked_flat_sorted = torch.cat(outs)   # B*T*K, C

out_flat_stacked_flat = torch.zeros(B*T*K, C, device=x.device, dtype=x.dtype)
out_flat_stacked_flat[indices_flat_sorted_indices] = out_flat_stacked_flat_sorted   # B*T*K, C
out_flat_stacked = out_flat_stacked_flat.reshape(B*T, K, C)
out_flat_stacked_weighted = out_flat_stacked * values.unsqueeze(-1)
out_flat = out_flat_stacked_weighted.sum(dim=1)   # B*T, C
outputs = out_flat.reshape(B, T, C)

In [38]:
outputs[0, :8, :5]

tensor([[-9.7591e-02,  1.0959e-01,  4.8590e-02,  1.6515e-01, -8.1924e-03],
        [ 2.1229e-01, -6.1279e-02, -6.9787e-02,  8.7662e-02, -1.1172e-01],
        [ 3.7798e-02, -2.2705e-01, -2.4165e-02, -2.9489e-02,  1.6072e-01],
        [ 5.8420e-01, -1.2971e-01,  3.7412e-02, -2.6841e-03, -1.1125e-02],
        [ 1.3185e-01,  2.2768e-01,  5.7638e-04,  8.3506e-02, -2.1502e-02],
        [-3.0618e-01,  7.0263e-02, -2.7878e-02,  1.8746e-02,  1.4195e-01],
        [-1.7710e-01,  1.8780e-01, -1.5720e-01,  4.0868e-01, -6.1299e-01],
        [-2.8894e-02,  1.1948e-01, -4.4672e-02, -8.1153e-02,  1.1719e-01]],
       grad_fn=<SliceBackward0>)

## Combined Version

In [39]:
class MyMoE(torch.nn.Module):
    def __init__(self, C, E, K):
        super().__init__()
        self.K = K
        self.router = nn.Linear(C, E, bias=False)
        self.experts_up = nn.ModuleList([nn.Linear(C, C*4, bias=False) for _ in range(E)])
        self.experts_down = nn.ModuleList([nn.Linear(C*4, C, bias=False) for _ in range(E)])
        self.expert_bias = nn.Parameter(torch.tensor([0.0, 0.1, 0.2, 0.3]))  # (E,)

    @torch.compiler.disable  # Dynamic slicing breaks the torch.compile
    def forward(self, x):
        B, T, C = x.shape
        K = self.K
        E = len(self.experts_up)
        x_flat = x.reshape(-1, C)          # B*T, C
        logits = self.router(x_flat)       # B*T, E
        weights = torch.sigmoid(logits)    # B*T, E
        # Bias the expert selection, but *not* weighting (Nanochat, DeepSeekV3)
        weights_biased = weights + self.expert_bias   # B*T, E
        _, indices = torch.topk(weights_biased, K, dim=-1)     # B*T, K
        values = torch.gather(weights, dim=-1, index=indices)   # B*T, K
        x_flat_stacked = torch.stack([x_flat]*K, dim=1)      # B*T, K, C
        x_flat_stacked_flat = x_flat_stacked.reshape(-1, C)  # B*T*K, C
        indices_flat = indices.reshape(-1)                   # B*T*K
        indices_flat_sorted_indices = torch.argsort(indices_flat, stable=True)  # B*T*K
        x_flat_stacked_flat_sorted = x_flat_stacked_flat[indices_flat_sorted_indices]  # B*T*K, C
        
        start_idx = 0
        outs = []
        for i in range(E):
            num_expert = (indices==i).sum().item()
            end_idx = start_idx + num_expert
            h = self.experts_up[i](x_flat_stacked_flat_sorted[start_idx:end_idx])
            z = F.relu(h).square()
            o = self.experts_down[i](z)
            outs.append(o)
            start_idx += num_expert
        out_flat_stacked_flat_sorted = torch.cat(outs)   # B*T*K, C
        
        out_flat_stacked_flat = torch.zeros(B*T*K, C, device=x.device, dtype=x.dtype)
        out_flat_stacked_flat[indices_flat_sorted_indices] = out_flat_stacked_flat_sorted   # B*T*K, C
        out_flat_stacked = out_flat_stacked_flat.reshape(B*T, K, C)
        out_flat_stacked_weighted = out_flat_stacked * values.unsqueeze(-1)
        out_flat = out_flat_stacked_weighted.sum(dim=1)   # B*T, C
        outputs = out_flat.reshape(B, T, C)
        return outputs

In [43]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)

B, T, C = 5, 16, 32
E = 4  # num experts
K = 2  # top_k

moe = MyMoE(C=C, E=E, K=K)

x = torch.randn(B, T, C)
outputs2 = moe(x)
loss2 = outputs2.float().square().mean()
loss2.backward()

In [44]:
outputs2[0, :8, :5]

tensor([[-9.7591e-02,  1.0959e-01,  4.8590e-02,  1.6515e-01, -8.1924e-03],
        [ 2.1229e-01, -6.1279e-02, -6.9787e-02,  8.7662e-02, -1.1172e-01],
        [ 3.7798e-02, -2.2705e-01, -2.4165e-02, -2.9489e-02,  1.6072e-01],
        [ 5.8420e-01, -1.2971e-01,  3.7412e-02, -2.6841e-03, -1.1125e-02],
        [ 1.3185e-01,  2.2768e-01,  5.7638e-04,  8.3506e-02, -2.1502e-02],
        [-3.0618e-01,  7.0263e-02, -2.7878e-02,  1.8746e-02,  1.4195e-01],
        [-1.7710e-01,  1.8780e-01, -1.5720e-01,  4.0868e-01, -6.1299e-01],
        [-2.8894e-02,  1.1948e-01, -4.4672e-02, -8.1153e-02,  1.1719e-01]],
       grad_fn=<SliceBackward0>)

In [45]:
outputs[0, :8, :5]

tensor([[-9.7591e-02,  1.0959e-01,  4.8590e-02,  1.6515e-01, -8.1924e-03],
        [ 2.1229e-01, -6.1279e-02, -6.9787e-02,  8.7662e-02, -1.1172e-01],
        [ 3.7798e-02, -2.2705e-01, -2.4165e-02, -2.9489e-02,  1.6072e-01],
        [ 5.8420e-01, -1.2971e-01,  3.7412e-02, -2.6841e-03, -1.1125e-02],
        [ 1.3185e-01,  2.2768e-01,  5.7638e-04,  8.3506e-02, -2.1502e-02],
        [-3.0618e-01,  7.0263e-02, -2.7878e-02,  1.8746e-02,  1.4195e-01],
        [-1.7710e-01,  1.8780e-01, -1.5720e-01,  4.0868e-01, -6.1299e-01],
        [-2.8894e-02,  1.1948e-01, -4.4672e-02, -8.1153e-02,  1.1719e-01]],
       grad_fn=<SliceBackward0>)